# 🛡️ AE-WGAN CICIDS-2017 — Version optimisée

- **Features** : Table 5 exacte (10 features ANOVA)
- **Architecture** : GitHub originale (BiGAN, activation **linear** pour CICIDS)
- **Reshape** : 2×5×1 (au lieu de 4×6 pour NSL-KDD)
- **Oversampling** : 10% de BENIGN
- **Classifieurs** : activation **tanh** (Table 4 CICIDS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers, regularizers
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Chargement CICIDS-2017

Les CSV doivent être dans `Google Drive/MyDrive/cicids2017/`

In [ ]:
CICIDS_DIR = '/content/drive/MyDrive/cicids2017'

if not os.path.exists(CICIDS_DIR):
    print(f"❌ Dossier non trouvé: {CICIDS_DIR}")
    print("Placez les CSV CICIDS-2017 dans Google Drive/MyDrive/cicids2017/")
else:
    csv_files = sorted([f for f in os.listdir(CICIDS_DIR) if f.endswith('.csv')])
    print(f"✅ {len(csv_files)} fichiers CSV trouvés:")
    for f in csv_files:
        size = os.path.getsize(os.path.join(CICIDS_DIR, f)) / 1e6
        print(f"  {f} ({size:.1f} MB)")

In [ ]:
print("Chargement des CSV...")
dfs = []
for f in csv_files:
    fpath = os.path.join(CICIDS_DIR, f)
    print(f"  {f}...", end=" ")
    try:
        df = pd.read_csv(fpath, encoding='utf-8', low_memory=False)
    except:
        df = pd.read_csv(fpath, encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.strip()  # Nettoyer espaces
    print(f"{df.shape[0]} rows")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all.columns = df_all.columns.str.strip()

# Nettoyer Label
if 'Label' in df_all.columns:
    label_col = 'Label'
elif 'label' in df_all.columns:
    label_col = 'label'
else:
    label_col = [c for c in df_all.columns if 'label' in c.lower()][0]

df_all[label_col] = df_all[label_col].str.strip()

print(f"\nDataset: {df_all.shape}")
print(f"\nDistribution:")
print(df_all[label_col].value_counts())

## 2. Preprocessing — Table 5 CICIDS-2017

In [ ]:
print("=" * 60)
print("PREPROCESSING CICIDS-2017")
print("=" * 60)

# Nettoyer inf/nan
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
n_before = len(df_all)
df_all.dropna(inplace=True)
print(f"Supprimé {n_before - len(df_all)} lignes NaN/Inf")

# Paper Table 5: 10 features ANOVA pour CICIDS-2017
PAPER_10_FEATURES_RAW = [
    'Destination Port', 'Flow Duration', 'Bwd Packet Length Min',
    'Flow IAT Mean', 'Flow IAT Max', 'Fwd IAT Mean',
    'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
    'min_seg_size_forward'
]

# Trouver les colonnes correspondantes (gérer les variations de noms)
available_cols = df_all.columns.tolist()
matched_features = []
for feat in PAPER_10_FEATURES_RAW:
    if feat in available_cols:
        matched_features.append(feat)
    else:
        # Chercher sans espaces ou avec underscore
        found = False
        for col in available_cols:
            if feat.replace(' ', '').lower() == col.replace(' ', '').lower():
                matched_features.append(col)
                found = True
                break
        if not found:
            print(f"  ⚠️ Feature '{feat}' non trouvée!")

print(f"\nFeatures matchées: {len(matched_features)}/10")
print(f"  {matched_features}")

# Labels
y_all = df_all[label_col].copy()

# Sélectionner les features
X_all = df_all[matched_features].copy()
X_all = X_all.apply(pd.to_numeric, errors='coerce')
X_all.dropna(inplace=True)
y_all = y_all.loc[X_all.index]

# Train/Test split stratifié 80/20
X_train_raw, X_test_raw, y_train_all, y_test_all = train_test_split(
    X_all.values, y_all.values, test_size=0.2, random_state=42, stratify=y_all)

# Min-Max Scaling
scaler = MinMaxScaler()
X_train_final = scaler.fit_transform(X_train_raw)
X_test_final = scaler.transform(X_test_raw)

y_train_cat = pd.Series(y_train_all)
y_test_cat = pd.Series(y_test_all)
y_train_detail = y_train_cat.copy()

INPUT_DIM = X_train_final.shape[1]  # = 10
N_BENIGN = (y_train_cat == 'BENIGN').sum()
TARGET = int(N_BENIGN * 0.10)

print(f"\n✅ Train: {X_train_final.shape}, Test: {X_test_final.shape}")
print(f"   INPUT_DIM = {INPUT_DIM}")
print(f"   BENIGN: {N_BENIGN}, Seuil 10%: {TARGET}")
print(f"\nDistribution (train):")
print(y_train_cat.value_counts())

## 3. AE-WGAN CICIDS-2017 (architecture GitHub — activation linear)

In [ ]:
class AEWGAN_CICIDS:
    """Code GitHub CICIDS: img_shape=(2,5,1), activation LINEAR partout."""
    def __init__(self):
        self.img_rows=2; self.img_cols=5; self.channels=1
        self.img_shape=(2,5,1); self.latent_dim=32
        self.flat_dim=int(np.prod(self.img_shape))  # =10
        opt=tf.keras.optimizers.Adamax(0.00009)
        self.discriminator=self._build_disc()
        self.discriminator.compile(loss=self._wloss,optimizer=opt,metrics=['accuracy'])
        self.generator=self._build_gen()
        self.encoder=self._build_enc()
        self.discriminator.trainable=False
        z=Input(shape=(self.latent_dim,)); img_=self.generator(z)
        img=Input(shape=self.img_shape); z_=self.encoder(img)
        fake=self.discriminator([z,img_]); valid=self.discriminator([z_,img])
        self.combined=Model([z,img],[fake,valid])
        self.combined.compile(loss=[self._wloss,self._wloss],optimizer=opt)

    def _wloss(self,y_true,y_pred): return K.mean(y_true*y_pred)

    def _build_enc(self):
        # CICIDS: activation LINEAR (pas relu)
        m=Sequential([Flatten(input_shape=self.img_shape),
                       Dense(20,activation='linear'),Dense(self.latent_dim)])
        img=Input(shape=self.img_shape); return Model(img,m(img),name='encoder')

    def _build_gen(self):
        # CICIDS: activation LINEAR partout (pas relu)
        m=Sequential([
            Dense(20,input_dim=self.latent_dim,activation='linear'),
            Dense(40,activation='linear'),Dense(80,activation='linear'),
            Dense(120,activation='linear'),Dense(160,activation='linear'),
            Dense(self.flat_dim,activation='linear'),Reshape(self.img_shape)])
        z=Input(shape=(self.latent_dim,)); return Model(z,m(z),name='generator')

    def _build_disc(self):
        # CICIDS: Dense linear puis sigmoid en sortie
        z=Input(shape=(self.latent_dim,)); img=Input(shape=self.img_shape)
        d_in=Concatenate()([z,Flatten()(img)])
        x=Dense(80,activation='linear')(d_in)
        x=Dense(40,activation='linear')(x)
        v=Dense(1,activation='sigmoid')(x)
        return Model([z,img],v,name='discriminator')

    def train(self,X,epochs=101,batch_size=32,verbose=1):
        reshaped=X.reshape(-1,self.img_rows,self.img_cols,self.channels)
        n=reshaped.shape[0]
        if n<batch_size: batch_size=max(n//2,2)
        valid=np.ones((batch_size,1)); fake=np.zeros((batch_size,1))
        for epoch in range(epochs):
            z=np.random.normal(0,1,size=(batch_size,self.latent_dim))
            imgs_=self.generator.predict(z,verbose=0)
            idx=np.random.randint(0,n,batch_size); imgs=reshaped[idx]
            z_=self.encoder.predict(imgs,verbose=0)
            self.discriminator.train_on_batch([z_,imgs],valid)
            self.discriminator.train_on_batch([z,imgs_],fake)
            self.combined.train_on_batch([z,imgs],[valid,fake])
            if verbose and (epoch+1)%max(1,epochs//5)==0:
                print(f"      Epoch {epoch+1}/{epochs}")

    def generate_batch(self,n):
        noise=np.random.normal(0,1,size=(n,self.latent_dim))
        return self.generator.predict(noise,verbose=0,batch_size=256).reshape(-1,self.flat_dim)

print("✅ AEWGAN_CICIDS défini (2×5, linear)")

## 4. Oversampling — 10% de BENIGN

In [ ]:
print("=" * 60)
print(f"AE-WGAN OVERSAMPLING — seuil: {TARGET}")
print("=" * 60)

attack_types = sorted([t for t in y_train_detail.unique() if t != 'BENIGN'])
print(f"Types d'attaques: {len(attack_types)}")

MIN_SAMPLES = 10
all_synthetic_X = []
all_synthetic_y = []
trained_models = {}
start_time = time.time()

for attack_name in attack_types:
    mask = y_train_detail == attack_name
    X_attack = X_train_final[mask.values]
    n_orig = X_attack.shape[0]

    if n_orig >= TARGET:
        print(f"  [{attack_name}] ⏩ {n_orig} >= {TARGET} — skip")
        continue
    if n_orig < MIN_SAMPLES:
        print(f"  [{attack_name}] ⏭️ {n_orig} < {MIN_SAMPLES} — skip")
        continue

    n_to_gen = TARGET - n_orig
    print(f"\n  [{attack_name}] 🧠 Training ({n_orig} samples, +{n_to_gen})...")

    model = AEWGAN_CICIDS()
    bs = min(32, max(n_orig // 2, 2))
    model.train(X_attack, epochs=101, batch_size=bs, verbose=1)
    synthetic = model.generate_batch(n_to_gen)
    all_synthetic_X.append(synthetic)
    all_synthetic_y.extend([attack_name] * n_to_gen)
    trained_models[attack_name] = model
    print(f"  [{attack_name}] ✅ {n_orig} + {n_to_gen} = {n_orig + n_to_gen}")

elapsed = (time.time() - start_time) / 60
print(f"\n--- {elapsed:.1f} minutes ---")

if all_synthetic_X:
    X_syn = np.vstack(all_synthetic_X)
    y_syn = pd.Series(all_synthetic_y)
    X_augmented = np.vstack([X_train_final, X_syn])
    y_augmented_cat = pd.concat([y_train_cat, y_syn], ignore_index=True)
else:
    X_augmented = X_train_final; y_augmented_cat = y_train_cat

print(f"\n  Original:    {X_train_final.shape[0]:>10}")
print(f"  Synthétique: {X_augmented.shape[0]-X_train_final.shape[0]:>10}")
print(f"  Total:       {X_augmented.shape[0]:>10}")

## 5. Classifieurs CICIDS-2017 (Table 4 — tanh)

In [ ]:
def build_dt(nc=2): return DecisionTreeClassifier(max_depth=3 if nc==2 else 10,random_state=42)
def build_rf(nc=2): return RandomForestClassifier(100,max_depth=9 if nc==2 else 7,random_state=42,n_jobs=-1)

def build_cnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='CNN'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.Conv1D(40,1,activation='tanh',padding='same'))
    if nc>2: m.add(layers.Conv1D(40,1,activation='tanh',padding='same'))
    m.add(layers.Flatten()); m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def build_gru(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.GRU(40,activation='tanh',return_sequences=True))
    m.add(layers.GRU(40,activation='tanh',return_sequences=True))
    m.add(layers.GRU(40,activation='tanh'))
    if nc==2:
        m.add(layers.Dense(40,activation='tanh',kernel_regularizer=regularizers.l1(0.005)))
    else:
        m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def build_lstm(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='LSTM'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.LSTM(40,activation='tanh',return_sequences=True))
    m.add(layers.LSTM(40,activation='tanh',return_sequences=True))
    m.add(layers.LSTM(40,activation='tanh'))
    m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def build_rnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='RNN'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.SimpleRNN(40,activation='tanh',return_sequences=True))
    m.add(layers.SimpleRNN(40,activation='tanh',return_sequences=True))
    m.add(layers.SimpleRNN(40,activation='tanh'))
    m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

print("✅ Classifieurs CICIDS-2017 (tanh)")

## 6. Classification

In [ ]:
def prepare_labels(y_tr,y_te,binary=True):
    le=LabelEncoder()
    if binary: return (y_tr!='BENIGN').astype(int).values,(y_te!='BENIGN').astype(int).values,le
    else:
        le.fit(list(set(y_tr.unique())|set(y_te.unique())))
        return le.transform(y_tr),le.transform(y_te),le

def train_and_eval(name,clf,X_tr,y_tr,X_te,y_te):
    if name in ['CNN','GRU','LSTM','RNN']:
        X_tr3=X_tr.reshape(-1,X_tr.shape[1],1); X_te3=X_te.reshape(-1,X_te.shape[1],1)
        es=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
        clf.fit(X_tr3,y_tr,epochs=50,batch_size=256,validation_split=0.1,callbacks=[es],verbose=0)
        yp=clf.predict(X_te3,verbose=0)
        y_pred=(yp.flatten()>0.5).astype(int) if yp.shape[-1]==1 else np.argmax(yp,axis=1)
    else: clf.fit(X_tr,y_tr); y_pred=clf.predict(X_te)
    avg='binary' if len(np.unique(y_te))==2 else 'weighted'
    return {'classifier':name,'accuracy':accuracy_score(y_te,y_pred),
            'precision':precision_score(y_te,y_pred,average=avg,zero_division=0),
            'recall':recall_score(y_te,y_pred,average=avg,zero_division=0),
            'f1_score':f1_score(y_te,y_pred,average=avg,zero_division=0)}

def run_all(X_tr,y_cat,X_te,y_te_cat,dim,binary=True):
    label="BINAIRE" if binary else "MULTICLASSE"
    print(f"\n{'='*60}")
    print(f"  CLASSIFICATION {label}")
    print(f"{'='*60}")
    y_tr,y_te,le=prepare_labels(y_cat,y_te_cat,binary)
    nc=2 if binary else len(np.unique(y_tr))
    print(f"  Classes: {nc}, Train: {X_tr.shape[0]}, Test: {X_te.shape[0]}")
    builders={'DT':lambda:build_dt(nc),'RF':lambda:build_rf(nc),
              'CNN':lambda:build_cnn(dim,nc),'GRU':lambda:build_gru(dim,nc),
              'LSTM':lambda:build_lstm(dim,nc),'RNN':lambda:build_rnn(dim,nc)}
    results=[]
    for name,builder in builders.items():
        print(f"  Training {name}...",end=" ")
        r=train_and_eval(name,builder(),X_tr,y_tr,X_te,y_te)
        print(f"Acc={r['accuracy']:.4f} F1={r['f1_score']:.4f}")
        results.append(r)
    return results

print("✅ Prêt")

In [ ]:
print("🔄 Baseline...")
base_bin=run_all(X_train_final,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,binary=True)
base_multi=run_all(X_train_final,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,binary=False)

print("\n🔄 AE-WGAN CICIDS-2017...")
enh_bin=run_all(X_augmented,y_augmented_cat,X_test_final,y_test_cat,INPUT_DIM,binary=True)
enh_multi=run_all(X_augmented,y_augmented_cat,X_test_final,y_test_cat,INPUT_DIM,binary=False)

## 7. Résultats

In [ ]:
print("\n" + "="*70)
print("  COMPARAISON BASELINE vs AE-WGAN — CICIDS-2017")
print("="*70)
for task,br,er in [("BINAIRE",base_bin,enh_bin),("MULTICLASSE",base_multi,enh_multi)]:
    print(f"\n  --- {task} ---")
    print(f"  {'Clf':<8} {'Base Acc':>9} {'AE Acc':>9} {'Δ Acc':>7}  {'Base F1':>9} {'AE F1':>9} {'Δ F1':>7}")
    print(f"  {'─'*60}")
    for b,e in zip(br,er):
        da=e['accuracy']-b['accuracy']; df=e['f1_score']-b['f1_score']
        print(f"  {b['classifier']:<8} {b['accuracy']:>9.4f} {e['accuracy']:>9.4f} {'+'if da>=0 else ''}{da:>6.4f}  "
              f"{b['f1_score']:>9.4f} {e['f1_score']:>9.4f} {'+'if df>=0 else ''}{df:>6.4f}")

# Visualization
fig,axes=plt.subplots(2,2,figsize=(16,12))
fig.suptitle('CICIDS-2017: Baseline vs AE-WGAN',fontsize=16,fontweight='bold')
for idx,(metric,title) in enumerate(zip(['accuracy','precision','recall','f1_score'],['Accuracy','Precision','Recall','F1 Score'])):
    ax=axes[idx//2][idx%2]
    clfs=[r['classifier'] for r in base_bin]; x=np.arange(len(clfs)); w=0.2
    ax.bar(x-1.5*w,[r[metric] for r in base_bin],w,label='Baseline Bin',color='#3498db',alpha=0.8)
    ax.bar(x-0.5*w,[r[metric] for r in enh_bin],w,label='AE-WGAN Bin',color='#e74c3c',alpha=0.8)
    ax.bar(x+0.5*w,[r[metric] for r in base_multi],w,label='Baseline Multi',color='#2ecc71',alpha=0.8)
    ax.bar(x+1.5*w,[r[metric] for r in enh_multi],w,label='AE-WGAN Multi',color='#f39c12',alpha=0.8)
    ax.set_xlabel('Classifieur');ax.set_ylabel(title);ax.set_title(title)
    ax.set_xticks(x);ax.set_xticklabels(clfs);ax.legend(fontsize=8);ax.set_ylim(0.3,1.05);ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('results_cicids2017.png',dpi=150,bbox_inches='tight')
plt.show()

# CSV + Drive
all_r=[]
for r in base_bin: all_r.append({**r,'type':'Baseline','classification':'Binary'})
for r in enh_bin: all_r.append({**r,'type':'AE-WGAN','classification':'Binary'})
for r in base_multi: all_r.append({**r,'type':'Baseline','classification':'Multiclass'})
for r in enh_multi: all_r.append({**r,'type':'AE-WGAN','classification':'Multiclass'})
df_r=pd.DataFrame(all_r)
df_r.to_csv('results_cicids2017.csv',index=False)
import shutil
shutil.copy('results_cicids2017.csv','/content/drive/MyDrive/')
shutil.copy('results_cicids2017.png','/content/drive/MyDrive/')
print("\n✅ Résultats sauvegardés et copiés sur Drive")
print(df_r.to_string(index=False))